<a href="https://colab.research.google.com/github/haze25102583/CNN/blob/main/day4_lesson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vision 모델 해석과 Grad-CAM
모델의 판단 근거

# [SECTION 01] 분석할 혼동 쌍 고르기
1. 혼동쌍→ 오답case·정답 control (=> 정답사례)

    confidence는 선정 기준

    판단 근거 : Grad-CAM, 가림 검증
2. target

    '어느 클래스 점수가 높아진 이유를 볼 것인가?'를 정하는 클래스 번호 -> 예측, 정답, 비교 후보
    Grad-CAM이 설명하는 클래스

3. 분석 레코드
    입력에 따라 CNN 내부에서 만든 feature maps를 확인

In [ ]:
# 고확신 오답, 정답 control 고르기

import numpy as np

y = np.array([0, 2, 0, 1])
p = np.array([
    [.80, .10, .10], [.72, .18, .10],
    [.20, .65, .15], [.10, .75, .15],
])
pred, conf = p.argmax(1), p.max(1)
wrong = np.flatnonzero(pred != y)             # np.flatnonzero: flatten(1차원)에서 0아닌 index값 반환
right = np.flatnonzero(pred == y)

case = wrong[conf[wrong].argmax()]
control = right[conf[right].argmax()]

print("case/control: ", case, control)

case/control:  1 0


# [SECTION 02] CNN 내부 반응

1. 필터

    모든 위치에 같은 가중치 ~> 특정 패턴 찾음
   
2. feature map

    한 이미지에서 그 패턴이 어디에 나타났는지 기록

3.  합성곱 출력

    (N, H, W, C) -> N=1이면, 하나의 사례 = 한 번에 처리하는 이미지 수
    Dense 뒤에는 HxW가 사라지므로 target layer로 쓸 수 x

    >**Grad-CAM** : 실제 입력의 feature maps에 target별 중요도 결합
    C를 가중합하고 HxW를 남겨 위치 지도를 만듦


4. 하나의 입력을 여러 층에 통과

5. 선택한 채널 활성화를 크게 만드는 합성 입력 만듦

    >**Gradient ascent(경사상승법)** : 어떤 값을 더 크게 만드는 방향으로 조금씩 이동하는 방법

6. 특정 target score에 기여한 공간을 클래스 별로 합침

    실제 입력의 마지막 합성곱 반응, target score gradient 사용

In [ ]:
# 채널을 임의로 고르는 것 방지
# absolute mean -> 관찰 후보를 정렬

import numpy as np

feature_maps = np.array([[
    [[0., 1., 0.], [1., 2., 0.]],
    [[0., 3., 1.], [0., 4., 1.]],
]])

strength = np.mean(
    np.abs(feature_maps), axis=(0, 1, 2)
)
top = np.argsort(strength)[::-1]

print("shape: ", feature_maps.shape)            # shape:  (1, 2, 2, 3)
print("strength: ", strength)                   # strength:  [0.25 2.5  0.5 ]
print("top: ", top)                             # top:  [1 2 0]

shape:  (1, 2, 2, 3)
strength:  [0.25 2.5  0.5 ]
top:  [1 2 0]


### 시각화

1. 필터 시각화

        모델이 학습한 모양, 패턴

2. 중간 활성화 시각화
        
        입력에 대해 반응한 층

3. Grad-CAM
        target 클래스 판단시 중요한 위치

가림(Occlusion): 이미지의특정부분을검은색·회색사각형등으로덮어서, CNN이그부분을보지못하게하는실험

# Grad-CAM

입력 이미지 -> 학습된 CNN(얕은 층 - 마지막 합성곱층 - 깊은 층) ->feature maps, class scores

마지막 합성곱 층의 freature maps, 최종 분류 결과인 class scores 동시 반환


### Grad-CAM 히트맵
**Grad-CAM** : 특징 지도의 반응 위치 x 선택한 클래스의 채널 중요도

**target 기준 채널 가중치** : 선택한 target_score를 feature_map로 미분

In [ ]:
# Grad-CAM

import numpy as np

# 한 이미지의 클래스별 예측 확률
probs = np.array([0.62, 0.28, 0.10])
true_target = 2                               # 실제 정답 클래스

# 확률이 높은 순서로 클래스 번호 정렬
ranked_targets = np.argsort(probs)[::-1]       # np.argsort() : 오름차순으로 정렬했을 때의 인덱스
                                               #                [0.62, 0.28, 0.10] -> [0.10, 0.28, 0.62] = [0, 1, 2]
                                               # [::-1] : 배열 뒤집기 -> 내림차순
pred_target = int(ranked_targets[0])           # 예측 1위
alt_target = int(ranked_targets[1])            # 비교할 2위 후보

# 3 가지 target을 한 곳에 기록
targets = {
    "pred" : pred_target,
    "true" : true_target,
    "alt" : alt_target,
}

# target 번호와 해당 클래스 score 출력
for name, target in targets.items():
  print(
      f"{name:>4}_target = {target}, "
      f"score = {probs[target]:.2f}"
  )

pred_target = 0, score = 0.62
true_target = 2, score = 0.10
 alt_target = 1, score = 0.28


MobileNetV2

    HxW가 남아있음. 분류 점수와 가까운 층 선택

GAP, Dense 뒤

    HxW가 사라짐 ->  Grad-CAM 지도 xx


얕은 층

    위치 선명, 약한 의미

마지막 Conv

    위치 유지, 의미 절충

Dense

    위치 소실, 클래스 점수

In [ ]:
# Funcional API -> feature maps, class scores

from tensorflow import keras
from tensorflow.keras import layers

inputs = keras.Input((8, 8, 1))
f = layers.Conv2D(                                # Conv2D : 특징 추출 -> feature maps
    4, 3, activation="relu", name="last_conv"
)(inputs)

x = layers.GlobalAveragePooling2D()(f)            # 평균값 -> 파라미터 단축
scores = layers.Dense(3)(x)                       # 클래스 예측 = class score
model = keras.Model(inputs, scores)

grad_model = keras.Model(                         # 모델의 기울기 -> CNN 복사x. 기존 graph에서 두 출력 지점 지정
    model.inputs, [f, scores]
)
print(grad_model.output_shape)

### Grad-CAM의 6단계

설명할 값 준비

1. **두 출력 경로 연결** : 기존 CNN에서 마지막 feature maps, 최종 class scores 반환 준비
2. **이미지 순전파** : -> target 선택



---

중요도 역추적

3. **기울기 계산** : feature maps가 조금 변할때, 점수가 얼마나 변하는 지
4. **채널 중요도** : 각 채널의 기울기 -> HxW 방향으로 평균 -> 중요도 숫자 1개 (요약)


---
히트맵 완성

5. **특징 지도 가중합** : 중요한 채널이면 더 강하게 반영한 위치 지도
6. **원본 이미지와 겹침** : 모델이 주목한 위치 확인


### GradientTape

Grad-CAM

    feature maps가 바뀌면, target score가 얼마나 변하는가

GradientTape

    계산을 잠시 기억 + 필요한 두 값 사이의 기울기


CODE


```
# 1. 계산 기록 시작
with tf.GradientTape() as tape:
  maps, scores = grad_model(
      img, training=False
  )
  target_score = scores[:, 0]

# 2. target score를 maps로 미분
grads = tape.gradient(target_score, maps)

```




In [ ]:
# GradientTape -> target score의 기울기

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

inputs = keras.Input((8,8,1))
f = layers.Conv2D(4, 3, activation="relu")(inputs)
scores = layers.Dense(3)(
    layers.GlobalAveragePooling2D()(f)
)
model = keras.Model(inputs, scores)
grad_model = keras.Model(model.inputs, [f, scores])
img = tf.ones((1, 8, 8, 1))

with tf.GradientTape() as tape:                         # with as : 자원을 열고, 실행이 끝나면 자원을 닫음
  maps, scores = grad_model(img, training=False)
  target_score = scores[:, 0]
grads = tape.gradient(target_score, maps)
print(maps.shape, grads.shape)                          # (1, 6, 6, 4) (1, 6, 6, 4)

(1, 6, 6, 4) (1, 6, 6, 4)


In [ ]:
# 공간 평균 -> 채널 중요도 1개씩
# 평균하는 값 = target score에 대한 gradient

import tensorflow as tf

grads = tf.reshape(
    tf.range(12, dtype=tf.float32), (1, 2, 2, 3),
)

weights = tf.reduce_mean(
    grads, axis=(0, 1, 2)
)

print(grads.shape, "->", weights.shape)
print(weights.numpy())

(1, 2, 2, 3) -> (3,)
[4.5 5.5 6.5]


In [ ]:
# feature maps x 채널 중요도 = 한 이미지 안에서의 상대적 양의 기여 강도
# 가중합, ReLU, 정규화 확인

import tensorflow as tf
maps = tf.constant([[
    [[1., 0.], [0., 1.]],
    [[2., 1.], [1., 0.]],
]])
weight = tf.constant([.8, -.2])                                               # 각 채널의 중요도

cam = tf.reduce_sum(
    maps[0] * weight, axis=-1                                                # axis=-1 -> Tensor의 마지막 차원
)

heatmap = tf.maximum(cam, 0)                                                  # 양의 기여만 (-> 이후 relu 활용)
heatmap /= tf.reduce_max(heatmap) + 1e-8                                      # 정규화. 1e-8 -> 분모 0을 방지
print("cam:\n", cam.numpy())
print("heatmap:\n", heatmap.numpy())

cam:
 [[ 0.8 -0.2]
 [ 1.4  0.8]]
heatmap:
 [[0.5714286 0.       ]
 [1.        0.5714286]]
